<a href="https://colab.research.google.com/github/aniket-alt/unsloth/blob/main/4_GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U unsloth transformers trl datasets accelerate peft bitsandbytes xformers gradio vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ==== System setup (Colab/Kaggle) ====

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
BF16 = is_bfloat16_supported() # autocasts to bf16 if available

# Utility: tiny evaluation helper
def chat(model, tokenizer, user, system="You are a helpful assistant.", max_new_tokens=128):
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    # 1. Tokenize. This returns a single PyTorch Tensor.
    inputs_tensor = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    )

    # 2. Move the tensor to the model's device
    inputs_on_device = inputs_tensor.to(model.device)

    from transformers import TextStreamer
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        # 3. Call generate.
        # We MUST pass `input_ids` as the first argument to avoid
        # a 'KeyError: key' bug in this version of Unsloth.
        _ = model.generate(
            input_ids = inputs_on_device, # <--- This must be the first argument
            streamer = streamer,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = 0.7
        )

print("Setup complete.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 11-11 02:35:14 [__init__.py:216] Automatically detected platform cuda.
ERROR 11-11 02:35:15 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
Setup complete.


In [3]:
# ==== D. GRPO for reasoning ====
from datasets import load_dataset
import re

# Model (smallish); GRPO works fine with LoRA for speed.
MODEL = "unsloth/Phi-3.5-mini-instruct-bnb-4bit"
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=64,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.11.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
# Dataset: GSM8K small split; we’ll force a "#### answer" format.
gsm = load_dataset("gsm8k", "main", split="train[:500]")

def to_prompts(batch):
    prompts, answers = [], []
    for q, a in zip(batch["question"], batch["answer"]):
        prompts.append([{"role":"user",
                         "content": f"{q}\n\nThink step by step. Give the final numeric answer after '####'."}])
        # Extract final numeric answer from GSM8K solution string
        m = re.findall(r"####\s*([-0-9\.,]+)", a)
        answers.append(m[-1].strip() if m else None)
    return {"prompt": prompts, "answer": answers}

gsm = gsm.map(to_prompts, batched=True, remove_columns=gsm.column_names)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [5]:
# Rewards: (1) exact "####" format presence; (2) exact numeric equality.
def reward_format(completions, **kwargs):
    scores=[]
    for comp in completions:
        text = comp[0]["content"]
        scores.append(3.0 if "####" in text else 0.0)
    return scores

def reward_exact(prompts, completions, answer, **kwargs):
    def extract_num(s):
        m = re.findall(r"####\s*([-0-9\.,]+)", s)
        return m[-1].strip() if m else None
    scores=[]
    for comp, true in zip(completions, answer):
        guess = extract_num(comp[0]["content"])
        scores.append(5.0 if (guess is not None and true is not None and guess==true) else -2.0)
    return scores

In [6]:
# Trainer config (small demo)
from trl import GRPOTrainer, GRPOConfig
from vllm import SamplingParams

sampling = SamplingParams(
    temperature=1.0, top_p=1.0, top_k=-1,
    stop=[tokenizer.eos_token], include_stop_str_in_output=True
)

cfg = GRPOConfig(
    vllm_sampling_params=sampling,
    learning_rate=5e-6, weight_decay=0.01, warmup_ratio=0.1, optim="adamw_8bit",
    per_device_train_batch_size=8, gradient_accumulation_steps=4,
    num_generations=4,                # #rollouts per prompt
    num_train_epochs=1,
    max_prompt_length=512,
    max_completion_length=256,
    logging_steps=5, output_dir="grpo-demo",
    bf16=BF16, fp16=not BF16, report_to="none",
)

In [7]:
# A tiny train/val split
split = gsm.train_test_split(test_size=0.02, seed=3407)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_format, reward_exact],
    args=cfg,
    train_dataset=split["train"],
    eval_dataset=split["test"],
)
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 490 | Num Epochs = 1 | Total steps = 61
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 59,768,832 of 3,880,848,384 (1.54% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / reward_format / mean,rewards / reward_format / std,rewards / reward_exact / mean,rewards / reward_exact / std
5,0.000000,3.837500,2.487000,212.456250,127.200000,256.000000,0.293750,194.500952,127.200000,251.600000,0,0,0,0,0,0.001721,2.118750,1.328842,1.718750,3.385011
10,0.000100,4.418750,1.932719,212.950000,127.200000,256.000000,0.281250,196.024695,127.200000,252.400000,No Log,No Log,No Log,No Log,No Log,0.139773,2.175000,1.319822,2.243750,3.439657
15,0.000500,4.387500,2.036674,205.043750,119.400000,256.000000,0.287500,183.497647,119.400000,242.200000,No Log,No Log,No Log,No Log,No Log,0.545801,2.231250,1.236222,2.156250,3.322035
20,0.000600,3.031250,2.792363,223.393750,143.400000,256.000000,0.418750,200.350284,143.400000,253.200000,No Log,No Log,No Log,No Log,No Log,0.613969,1.837500,1.447552,1.193750,3.503600
25,0.001300,5.062500,2.405250,203.075000,115.200000,256.000000,0.231250,187.227896,115.200000,253.000000,No Log,No Log,No Log,No Log,No Log,1.322647,2.381250,1.219479,2.681250,3.279758
30,0.001300,4.037500,2.041987,211.237500,134.000000,256.000000,0.306250,194.123969,134.000000,249.400000,No Log,No Log,No Log,No Log,No Log,1.306599,2.100000,1.252585,1.937500,3.200928
35,0.001400,3.262500,2.826176,216.175000,131.400000,256.000000,0.362500,194.207150,131.400000,250.600000,No Log,No Log,No Log,No Log,No Log,1.352575,2.025000,1.412549,1.237500,3.470763
40,0.001800,3.975000,2.221762,207.456250,126.600000,256.000000,0.300000,187.331427,126.600000,254.200000,No Log,No Log,No Log,No Log,No Log,1.795799,2.081250,1.373513,1.893750,3.411098
45,0.002200,3.775000,3.076743,197.150000,116.200000,256.000000,0.218750,181.258414,116.200000,245.000000,No Log,No Log,No Log,No Log,No Log,2.152336,2.231250,1.317525,1.543750,3.512026
50,0.002200,5.018750,2.534571,199.187500,125.400000,255.800000,0.218750,182.877475,125.400000,247.600000,No Log,No Log,No Log,No Log,No Log,2.165062,2.381250,1.156306,2.637500,3.238402


TrainOutput(global_step=61, training_loss=0.0012842992198516111, metrics={'train_runtime': 7113.6575, 'train_samples_per_second': 0.069, 'train_steps_per_second': 0.009, 'total_flos': 0.0, 'train_loss': 0.0012842992198516111})

In [8]:
FastLanguageModel.for_inference(model)
print("\n=== Test (GRPO) ===")
chat(model, tokenizer, "A shop sells 3 pencils for $2. How much do 12 pencils cost? End with '#### <number>'.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== Test (GRPO) ===
To find out the cost of 12 pencils, we first need to determine the cost of one pencil.

Given that 3 pencils cost $2, we can calculate the cost of one pencil as follows:

Cost of 3 pencils = $2
Cost of 1 pencil = $2 / 3

Now, to find the cost of 12 pencils, we multiply the cost of one pencil by 12:

Cost of 12 pencils = (Cost of 1 pencil


In [9]:
# ==== Minimal Gradio chat ====
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread
FastLanguageModel.for_inference(model)

def respond(message, history):
    msgs = []
    for u,a in history + [(message,"")]:
        msgs.append({"role":"user","content":u})
        msgs.append({"role":"assistant","content":a})
    msgs.pop()  # drop trailing empty assistant
    input_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    def run_gen(): model.generate(input_ids=input_ids, streamer=streamer, max_new_tokens=256, temperature=0.7, do_sample=True)
    Thread(target=run_gen).start()
    partial=""
    for token in streamer:
        partial += token
        yield partial

gr.ChatInterface(respond, title="Unsloth Chat UI").launch(share=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>